# 📝 การบ้าน Day 2: Building Agents with Google ADK

**เกณฑ์การให้คะแนน: 10 คะแนน**

| ขั้นตอน | คะแนน | เนื้อหา |
|---------|-------|--------|
| 1. Agent + Custom Tool | 3 | สร้าง Agent พร้อม tool ที่ทำงานได้จริง |
| 2. RAG Agent | 3 | ต่อ Agent เข้ากับ Qdrant ค้นหาข้อมูลเฉพาะตัว |
| 3. Workflow Agent | 2 | ใช้ Sequential/Parallel/Loop สร้าง pipeline |
| 4. อธิบายผลลัพธ์ | 2 | อธิบายด้วยตัวเอง |

### ⚠️ กฎ
- ข้อมูลสร้างจาก **รหัสนักศึกษา** → ผลลัพธ์แต่ละคนไม่เหมือนกัน
- **ห้ามแก้ไข** cell ที่ระบุ
- ต้อง **Run ทุก cell** ให้มีผลลัพธ์ก่อนส่ง

## 📦 ติดตั้ง Dependencies

In [ ]:
%%time
!pip install -q google-adk google-genai sentence-transformers qdrant-client langchain-text-splitters

import hashlib, os, json, random
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings('ignore')

print('✅ ติดตั้งเรียบร้อย!')

In [ ]:
%%time
# ตั้งค่า Gemini API
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ โหลด API Key จาก Colab Secrets')
except Exception:
    api_key = input('🔑 กรุณาวาง Gemini API Key: ')
    os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'
print('✅ API Key ตั้งค่าเรียบร้อย')

## 🎓 กรอกข้อมูลนักศึกษา

In [ ]:
# ─── กรอกข้อมูลของคุณ ───
STUDENT_NAME = ''   # เช่น 'สมชาย ใจดี'
STUDENT_ID   = ''   # เช่น '6512345678'
PHONE        = ''   # เช่น '081-234-5678'
LINE_ID      = ''   # เช่น 'somchai.j'

# ─── ตรวจสอบ (ห้ามแก้ไข) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข)

> ข้อมูลสร้างจากรหัสนักศึกษา → แต่ละคนได้ข้อมูลไม่เหมือนกัน

In [ ]:
# ===== ห้ามแก้ไข cell นี้ =====
random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_topics = [
    {'topic': 'healthcare', 'text': 'โรงพยาบาลศิริราช นำ AI มาช่วยวิเคราะห์ภาพ X-ray ทรวงอก ลดเวลาวินิจฉัยจาก 30 นาทีเหลือ 5 นาที ระบบ Deep Learning ตรวจจับความผิดปกติได้แม่นยำ 95% ช่วยแพทย์คัดกรองผู้ป่วยวัณโรคได้เร็วขึ้น'},
    {'topic': 'banking', 'text': 'ธนาคารกสิกรไทย พัฒนา KADE AI Assistant ใช้ RAG ดึงข้อมูลผลิตภัณฑ์ทางการเงินมาตอบลูกค้า ลดเวลาตอบคำถามจาก 5 นาทีเหลือ 30 วินาที รองรับภาษาไทยและอังกฤษ'},
    {'topic': 'education', 'text': 'จุฬาลงกรณ์มหาวิทยาลัย สร้างระบบ RAG ถาม-ตอบจากตำราเรียน แปลง PDF กว่า 500 เล่มเป็น vector embeddings ใช้ multilingual model รองรับภาษาไทย'},
    {'topic': 'agriculture', 'text': 'กรมวิชาการเกษตร ใช้ AI วิเคราะห์ภาพถ่ายโดรน ตรวจจับโรคพืชในนาข้าว ความแม่นยำ 92% ช่วยลดความเสียหายจากโรคพืชได้ 40%'},
    {'topic': 'logistics', 'text': 'Kerry Express ใช้ AI วางแผนเส้นทางขนส่งพัสดุ ลดระยะทางขนส่ง 15% ประหยัดน้ำมัน 200 ล้านบาทต่อปี ใช้ Machine Learning ทำนายปริมาณพัสดุล่วงหน้า'},
    {'topic': 'retail', 'text': 'เซ็นทรัล รีเทล ใช้ AI Recommendation System แนะนำสินค้าให้ลูกค้า ยอดขายเพิ่ม 25% วิเคราะห์พฤติกรรมการซื้อจากข้อมูลกว่า 10 ล้านรายการ'},
    {'topic': 'energy', 'text': 'การไฟฟ้าฝ่ายผลิต (EGAT) ใช้ AI พยากรณ์ความต้องการไฟฟ้า ลดการสูญเสียพลังงาน 8% ใช้ Deep Learning วิเคราะห์ข้อมูลจาก Smart Meter ทั่วประเทศ'},
    {'topic': 'tourism', 'text': 'ททท. พัฒนา AI Chatbot ให้ข้อมูลท่องเที่ยวไทย รองรับ 5 ภาษา ให้บริการนักท่องเที่ยวกว่า 1 ล้านคนต่อเดือน ลดภาระ call center ได้ 60%'},
    {'topic': 'insurance', 'text': 'เมืองไทยประกันชีวิต ใช้ AI ประเมินความเสี่ยงสุขภาพ วิเคราะห์ข้อมูลการเคลม ลดเวลาพิจารณาจาก 7 วันเหลือ 1 วัน ตรวจจับ fraud ได้ 90%'},
    {'topic': 'manufacturing', 'text': 'SCG ใช้ AI ตรวจสอบคุณภาพสินค้า Computer Vision ตรวจจับข้อบกพร่อง ลดของเสีย 30% Predictive Maintenance ทำนายการเสียของเครื่องจักรล่วงหน้า 2 สัปดาห์'},
]

random.shuffle(all_topics)
my_topics = all_topics[:6]  # เลือก 6 จาก 10 หัวข้อ

# เตรียมข้อมูลสำหรับ Qdrant
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)

my_chunks = []
my_sources = []
for item in my_topics:
    chunks = splitter.split_text(item['text'])
    for c in chunks:
        my_chunks.append(c)
        my_sources.append(item['topic'])

# Embed + Index
passages = [f'passage: {c}' for c in my_chunks]
embeddings = embed_model.encode(passages)
VECTOR_SIZE = embeddings.shape[1]

qdrant = QdrantClient(':memory:')
collection_name = f'hw2_{STUDENT_ID}'
qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

points = [
    models.PointStruct(id=i, vector=embeddings[i].tolist(),
                       payload={'text': my_chunks[i], 'source': my_sources[i]})
    for i in range(len(my_chunks))
]
qdrant.upsert(collection_name=collection_name, points=points)

# สร้าง query เฉพาะตัว
query_pool = [
    'AI ช่วยลดต้นทุนในธุรกิจอย่างไร',
    'ระบบอัตโนมัติในองค์กรไทย',
    'การวิเคราะห์ข้อมูลด้วย Machine Learning',
    'Chatbot สำหรับบริการลูกค้า',
    'Computer Vision ในอุตสาหกรรมไทย',
    'การพยากรณ์ด้วย Deep Learning',
]
random.shuffle(query_pool)
MY_QUERY = query_pool[0]

# เลือก tool type
tool_types = [
    {'name': 'คำนวณ ROI ของการลงทุน AI', 'func': 'calculate_ai_roi', 'params': 'investment_baht: float, savings_per_year: float, years: int'},
    {'name': 'ประเมินความพร้อม AI', 'func': 'assess_ai_readiness', 'params': 'num_employees: int, has_data: bool, budget_million: float'},
    {'name': 'คำนวณ Accuracy Score', 'func': 'calculate_accuracy', 'params': 'true_positives: int, false_positives: int, false_negatives: int, true_negatives: int'},
    {'name': 'แนะนำ AI Model', 'func': 'recommend_model', 'params': 'task_type: str, data_size: str, budget: str'},
]
random.shuffle(tool_types)
MY_TOOL = tool_types[0]

# Workflow type
workflow_types = ['SequentialAgent', 'ParallelAgent', 'LoopAgent']
random.shuffle(workflow_types)
MY_WORKFLOW = workflow_types[0]

print(f'✅ ข้อมูลเฉพาะตัวพร้อม!')
print(f'   📊 หัวข้อ: {[t["topic"] for t in my_topics]}')
print(f'   📦 Chunks: {len(my_chunks)} | Collection: {collection_name}')
print(f'   🔍 Query ที่ต้องใช้: "{MY_QUERY}"')
print(f'   🔧 Tool ที่ต้องสร้าง: {MY_TOOL["name"]}')
print(f'   🔄 Workflow ที่ต้องใช้: {MY_WORKFLOW}')

## 🔧 ฟังก์ชันช่วย (ห้ามแก้ไข)

In [ ]:
# ===== ห้ามแก้ไข cell นี้ =====
async def chat_with_agent(agent, message, user_id='user_1', session_id=None):
    """ส่งข้อความถึง Agent แล้วรับคำตอบ"""
    if session_id is None:
        session_id = f'session_{id(agent)}'
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    session = await runner.session_service.create_session(
        app_name=agent.name, user_id=user_id, session_id=session_id)
    content = types.Content(role='user', parts=[types.Part.from_text(text=message)])

    final_response = ''
    tool_calls = []
    async for event in runner.run_async(
        user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text: final_response = part.text
                if part.function_call:
                    tool_calls.append(part.function_call.name)

    if tool_calls:
        print(f'  🔧 Tools: {tool_calls}')
    return final_response, session.id

print('✅ ฟังก์ชัน chat พร้อม')


---
## 🎯 ขั้นตอนที่ 1: สร้าง Agent + Custom Tool (3 คะแนน)

สร้าง Agent ที่มี **2 tools**:
1. `search_knowledge` — ค้นหาข้อมูลจาก Qdrant (โค้ดให้แล้ว)
2. **Custom Tool** — ตามที่ระบบกำหนดให้ (ดูจาก `MY_TOOL`)

### เกณฑ์
- 1 คะแนน: สร้าง custom tool ที่ทำงานได้จริง (มี docstring + type hints + return dict)
- 1 คะแนน: สร้าง Agent ที่ใช้ทั้ง 2 tools ได้
- 1 คะแนน: ทดสอบให้เห็นว่า Agent เลือก tool ถูกต้อง

In [ ]:
# RAG Tool (ให้แล้ว — ห้ามแก้)
def search_knowledge(query: str) -> dict:
    """ค้นหาข้อมูลเกี่ยวกับ AI ในประเทศไทยจากฐานความรู้
    
    Args:
        query: คำถามที่ต้องการค้นหา
    """
    q_vec = embed_model.encode(f'query: {query}').tolist()
    results = qdrant.query_points(
        collection_name=collection_name, query=q_vec, limit=3).points
    return {
        'status': 'success',
        'results': [{'text': r.payload['text'], 'source': r.payload['source'],
                     'score': round(r.score, 4)} for r in results]
    }

print('✅ RAG Tool พร้อม')
print(f'\n📌 Tool ที่คุณต้องสร้าง: {MY_TOOL["name"]}')
print(f'   ฟังก์ชัน: {MY_TOOL["func"]}({MY_TOOL["params"]})')
print(f'   ต้อง return dict')

In [ ]:
# ขั้นตอนที่ 1: เขียนโค้ดที่นี่

# 💡 Hint:
#   1. สร้างฟังก์ชันตามที่ MY_TOOL กำหนด
#   2. ต้องมี docstring (LLM อ่าน!)
#   3. ต้องมี type hints
#   4. ต้อง return dict

# def MY_TOOL['func'](...) -> dict:
#     """อธิบาย..."""
#     ...
#     return {...}



# สร้าง Agent พร้อม 2 tools
# my_agent = Agent(
#     name='my_agent',
#     model='gemini-2.5-flash',
#     tools=[search_knowledge, my_custom_tool]
# )



# ทดสอบ Tool Selection (ถามอย่างน้อย 3 คำถาม)
# 1. คำถามที่ต้องใช้ RAG tool
# 2. คำถามที่ต้องใช้ custom tool
# 3. คำถามทั่วไป (ไม่ต้องใช้ tool)
# await chat_with_agent(my_agent, '...')

---
## 🎯 ขั้นตอนที่ 2: RAG Agent — ค้นหาข้อมูลเฉพาะตัว (3 คะแนน)

ใช้ Agent ค้นหาข้อมูลจาก Qdrant ด้วย **query ที่ระบบกำหนด**

### เกณฑ์
- 1 คะแนน: Agent ค้นหาได้ + แสดง Top-3 scores
- 1 คะแนน: คำตอบอ้างอิงข้อมูลจากฐานความรู้
- 1 คะแนน: อธิบายด้วยตัวเองว่าทำไม chunk เหล่านี้ถูกค้นพบ (3-5 ประโยค)

In [ ]:
# ขั้นตอนที่ 2: เขียนโค้ดที่นี่

print(f'🔍 Query ที่ต้องใช้: "{MY_QUERY}"')

# 💡 Hint:
#   1. สร้าง Agent ที่มี instruction ให้ค้นหาจาก search_knowledge
#   2. ถามด้วย MY_QUERY
#   3. แสดง Top-3 scores
#   4. เขียนอธิบายด้วยตัวเอง (ไม่ใช้ AI)

# rag_agent = Agent(
#     name='rag_agent',
#     model='gemini-2.5-flash',
#     instruction='...',
#     tools=[search_knowledge]
# )
# response, _ = await chat_with_agent(rag_agent, MY_QUERY)
# print(response)

In [ ]:
# ขั้นตอนที่ 2 (ต่อ): อธิบายด้วยตัวเอง
# เขียนเป็นข้อความ 3-5 ประโยค อธิบายว่า:
# 1. ทำไม chunks เหล่านี้ถูกค้นพบ?
# 2. ข้อมูลที่ค้นพบเกี่ยวข้องกับ query อย่างไร?
# 3. ถ้าจะปรับปรุงให้ค้นหาได้ดีขึ้น ต้องทำอย่างไร?

MY_EXPLANATION_STEP2 = '''
(เขียนคำอธิบายที่นี่)
'''
print(MY_EXPLANATION_STEP2)

---
## 🎯 ขั้นตอนที่ 3: Workflow Agent (2 คะแนน)

สร้าง Workflow ตาม **pattern ที่ระบบกำหนด** ดูจาก `MY_WORKFLOW`

### เกณฑ์
- 1 คะแนน: สร้าง Workflow Agent ได้ถูก pattern
- 1 คะแนน: ทดสอบแล้วทำงานได้จริง + แสดงผลลัพธ์

In [ ]:
# ขั้นตอนที่ 3: เขียนโค้ดที่นี่

print(f'🔄 Workflow ที่ต้องใช้: {MY_WORKFLOW}')

# 💡 Hint ตาม pattern:
#
# ถ้า SequentialAgent:
#   from google.adk.agents import SequentialAgent
#   pipeline = SequentialAgent(sub_agents=[agent1, agent2, agent3])
#   → ค้น → สรุป → แปล (ทีละขั้น)
#
# ถ้า ParallelAgent:
#   from google.adk.agents import ParallelAgent
#   parallel = ParallelAgent(sub_agents=[search1, search2, search3])
#   → ค้นหลายหัวข้อพร้อมกัน
#
# ถ้า LoopAgent:
#   from google.adk.agents import LoopAgent
#   loop = LoopAgent(sub_agents=[writer, critic], max_iterations=3)
#   → เขียน ↔ ตรวจ จนผ่าน



# ทดสอบ
# response, _ = await chat_with_agent(my_workflow, MY_QUERY)
# print(response)

---
## 🎯 ขั้นตอนที่ 4: อธิบายผลลัพธ์ (2 คะแนน)

### เกณฑ์
- 1 คะแนน: อธิบาย Workflow ที่เลือก — ทำไมเหมาะกับงาน? ข้อดี/ข้อเสีย?
- 1 คะแนน: เปรียบเทียบกับอีก 2 patterns — ถ้าใช้ pattern อื่นจะต่างอย่างไร?

In [ ]:
# ขั้นตอนที่ 4: เขียนคำอธิบาย

MY_EXPLANATION_STEP4 = f'''
Pattern ที่ใช้: {MY_WORKFLOW}

1. ทำไม {MY_WORKFLOW} เหมาะกับงานนี้:
   (เขียนที่นี่)

2. ข้อดีของ {MY_WORKFLOW}:
   (เขียนที่นี่)

3. ข้อเสียของ {MY_WORKFLOW}:
   (เขียนที่นี่)

4. ถ้าใช้ pattern อื่นจะต่างอย่างไร:
   (เขียนที่นี่)
'''
print(MY_EXPLANATION_STEP4)

## ✅ ตรวจสอบคำตอบ

In [ ]:
# ===== ห้ามแก้ไข cell นี้ =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_day2_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'🎓 รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 ส่งก่อน: _________________ (อาจารย์กำหนด)')
print('=' * 50)
print()
print(f'📌 ข้อมูลเฉพาะตัว:')
print(f'   หัวข้อ: {[t["topic"] for t in my_topics]}')
print(f'   Query: {MY_QUERY}')
print(f'   Tool: {MY_TOOL["name"]}')
print(f'   Workflow: {MY_WORKFLOW}')
print()
print('📋 Checklist ก่อนส่ง:')
print('  [ ] กรอกข้อมูลส่วนตัวครบถ้วน')
print('  [ ] ขั้นตอนที่ 1: สร้าง Agent + Custom Tool + ทดสอบ 3 คำถาม')
print('  [ ] ขั้นตอนที่ 2: RAG Agent + Top-3 scores + อธิบาย 3-5 ประโยค')
print('  [ ] ขั้นตอนที่ 3: Workflow Agent ตาม pattern ที่กำหนด')
print('  [ ] ขั้นตอนที่ 4: อธิบายผลลัพธ์ + เปรียบเทียบ')
print('  [ ] ทุก cell run แล้วมีผลลัพธ์')